# Sultai bounded local search

Uses the supplied 60-card deck, Standard-filtered UBG lands, four fixed Caverns and one fixed Swamp. Opulent Palace ranges from zero to two. Three deterministic starts share exact component caches; budgets total at most 500 visited bases. No global grid is constructed. Results are best found, with each run's stopping reason shown.

Incarnation profiles measure access to enters effects. Spider-Man includes copying payoff scenarios; Sunderflock assumes no Elemental, MV5, or MV6. Mode weights are uniform. Overlord uses impending 1B. V1 availability and unrestricted Cavern production are optimistic approximations; no board-state sequencing is simulated.


In [ ]:
from pathlib import Path
import sys
from collections import defaultdict
from html import escape
from IPython.display import HTML, display

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mana_search.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from examples.sultai import LAND_NAMES, MANA_BASE, SPELLS
from examples.sultai_optimize import (
    LAND_COUNT, COLORS, FORMAT, LAND_FAMILIES, FIXED_LANDS, TURNS, POSITION,
    OPENING_HAND_SIZE, EVALUATION_BUDGETS, SHOCK_PENALTY, TOWN_PENALTY, optimize,
)
from mana_lands import ManaSource
from mana_search import ManaSpace, LocalManaEvaluator, local_search

def name(source):
    return LAND_NAMES.get(source, f'{source.kind}:{source.colors}')

def lands(entries):
    return ', '.join(f'{n} {name(s)}' for s, n in entries) or 'none'

def show(headers, rows):
    head = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{head}</tr></thead><tbody>{body}</tbody></table>'))

## Domain and structured evaluation

The domain uses the catalog's dated Standard snapshot; it does not fetch current legality.
Four Caverns and one Swamp occupy five of the 26 slots. Source bounds define eligibility;
the starting base is supplied explicitly and is validated before any search. Optional source
families can be added here. Opportunities remain split by casting cost, turn, and play/draw;
mana value is a reporting grouping, so different colored costs remain distinct.

In [ ]:
INITIAL_BASE = dict(MANA_BASE)
space = ManaSpace.from_colors(LAND_COUNT, COLORS, families=LAND_FAMILIES,
                             format=FORMAT, fixed=FIXED_LANDS)
space.counts(INITIAL_BASE)
evaluator = LocalManaEvaluator(space, SPELLS, turns=TURNS, position=POSITION,
                               opening_hand_size=OPENING_HAND_SIZE)
show(['Source', 'Minimum', 'Maximum', 'Initial'],
     [(name(s), lo, hi, INITIAL_BASE.get(s, 0)) for s, (lo, hi) in space.bounds.items()])
print(f'{LAND_COUNT} lands; {len(space.sources)} source types. No global grid constructed.')


# Sultai bounded local search

Uses the supplied 60-card deck, Standard-filtered UBG lands, four fixed Caverns and one fixed Swamp. Opulent Palace ranges from zero to two. Three deterministic starts share exact component caches; budgets total at most 500 visited bases. No global grid is constructed. Results are best found, with each run's stopping reason shown.

Incarnation profiles measure access to enters effects. Spider-Man includes copying payoff scenarios; Sunderflock assumes no Elemental, MV5, or MV6. Mode weights are uniform. Overlord uses impending 1B. V1 availability and unrestricted Cavern production are optimistic approximations; no board-state sequencing is simulated.


In [ ]:
# Edit budgets or penalties here; component caches survive penalty-only changes.
SHOCK_PENALTY = 0.012
TOWN_PENALTY = 0.015
EVALUATION_BUDGETS = (200, 150, 150)

cached_before = len(evaluator.cache)
runs, winner = optimize(
    evaluator, budgets=EVALUATION_BUDGETS,
    shock_penalty=SHOCK_PENALTY, town_penalty=TOWN_PENALTY,
    progress=lambda i, n, best: print(
        f'Start {i+1}: {n:4d} bases visited' + (f'; best {float(best.score):.8f}' if best else '')),
)
baseline = evaluator.value(INITIAL_BASE, shock_penalty=SHOCK_PENALTY, town_penalty=TOWN_PENALTY)
print(f'{len(evaluator.cache) - cached_before} new draw evaluations; {sum(r.seconds for r in runs):.2f} seconds')
show(['Start', 'Budget', 'Visits', 'Stop', 'Best score'],
     [(i+1, b, r.evaluations, r.stopped, float(r.best.score))
      for i, (b, r) in enumerate(zip(EVALUATION_BUDGETS, runs))])
show(['Base', 'Raw opportunities', 'Penalty', 'Score', 'Delta from supplied base'], [
    ('Initial', float(baseline.raw_score), float(baseline.penalty), float(baseline.score), 0),
    ('Best found', float(winner.raw_score), float(winner.penalty), float(winner.score), float(winner.score-baseline.score)),
])
show(['Source', 'Initial', 'Result', 'Change'],
     [(name(s), a, b, b-a) for s, a, b in zip(space.sources, baseline.counts, winner.counts)])


## Opportunity changes

These tables describe the result versus the starting base. Expected copies retain the
spell-mode shares. Weighted contributions include spell importance; they exclude composition
penalties. Every cell contribution is divided by the total number of turn/position cells,
so summing a table's weighted deltas reproduces the raw-score change. Play and draw are
also shown separately. Cost groups retain their mana value as an additional label.

In [ ]:
before = evaluator.components(INITIAL_BASE)
after = evaluator.components(dict(zip(space.sources, winner.counts)))
divisor = TURNS * (2 if POSITION == 'both' else 1)

def mana_value(cost):
    return int(cost.rstrip('WUBRG') or 0) + sum(cost.count(c) for c in 'WUBRG')

def comparison(key, headers):
    groups = defaultdict(lambda: [0.0, 0.0, 0.0, 0.0])
    for index, components in enumerate((before, after)):
        for row in components.opportunities:
            values = groups[key(row)]
            values[index] += float(row.expected_copies) / divisor
            values[index + 2] += row.weighted_contribution / divisor
    show(headers + ['Initial copies', 'Result copies', 'Copy delta', 'Weighted delta'],
         [(*k, f'{v[0]:.8f}', f'{v[1]:.8f}', f'{v[1]-v[0]:+.8f}', f'{v[3]-v[2]:+.8f}')
          for k, v in sorted(groups.items())])

comparison(lambda r: (mana_value(r.cost), r.cost), ['MV', 'Cost'])
comparison(lambda r: (mana_value(r.cost),), ['MV'])
comparison(lambda r: (r.turn, r.position), ['Turn', 'Position'])
assert winner.score >= baseline.score
assert sum(r.evaluations for r in runs) <= sum(EVALUATION_BUDGETS)
assert abs(sum(r.weighted_contribution for r in after.opportunities) / divisor
           - float(winner.raw_score)) < 1e-12